In [40]:
# Custom
import sys
sys.path.insert(0, '../src')
from args import args
from static_analyzer_path_resolver import resolve_path
import utils


## Utils and Inference
import csv
from collections import Counter
import json
from numpy import nan, NAN, NaN 
import pandas as pd
import re

import os

from tqdm import tqdm

In [ ]:
class args(args):
    code_dir = "../data/codes/"


    annotation_set = "../data/get_all_variable_info_info_annotation_set_23_06_2026_with_path.json"
    


    setting = ["batching", "single_variable"][0]

    batch_criteria = ["Father"][0]

    batch_method = ["record"][0]

    type_information = ["var_occ", "para_occ"][0]

    # Data from static analysis for both para and variable occ
    para_data_path = {
        "para_occ":'../data/get_variable_para_info_formatted_info_annotation_set_23_06_2026_with_path.json', 
        "var_occ": '../data/get_all_variable_info_info_annotation_set_23_06_2026_with_path.json'
    }[type_information]

    complete_data_path = {
        "para_occ":'../data/static_analyzer_Downloaded_Data/all_application_get_variable_para_info_formatted_23_06_2026.json',
        "var_occ": '../data/static_analyzer_Downloaded_Data/all_application_get_all_variable_info_23_06_2026.json'
    }[type_information]

    variable_column = "variable_name"

## Load Data

In [12]:
# args.complete_data_path, args.para_data_path

In [43]:
excluded_descriptors = ['ASM: ASM_B',
 'ASM: ASM_BE',
 'ASM: ASM_BR',
 'ASM: ASM_CLC',
 'ASM: ASM_CLI',
 'ASM: ASM_DROP',
 'ASM: ASM_L',
 'ASM: ASM_LA',
 'ASM: ASM_LM',
 'ASM: ASM_LR',
 'ASM: ASM_MACRO_CALL',
 'ASM: ASM_MVC',
 'ASM: ASM_MVI',
 'ASM: ASM_SR',
 'ASM: ASM_ST',
 'ASM: ASM_STM',
 'ASM: ASM_USING']

In [ ]:
# para_data_path: aggregated per-variable rows, list-valued columns for multi-occurrence fields
para_data_df = pd.read_json(args.para_data_path)
para_data_df['corrected_path'] = para_data_df['PATHSTR'].apply(resolve_path)
para_data_df.columns

In [44]:
# complete_data_path: flat table, one row per variable occurrence across all applications
complete_data_df = pd.read_json(args.complete_data_path)
complete_data_df['corrected_path'] = complete_data_df['PATHSTR'].apply(resolve_path)
complete_data_df.columns

if "StatementDescription" in complete_data_df.columns:
    complete_data_df = complete_data_df[~complete_data_df['StatementDescription'].isin(excluded_descriptors)].reset_index(drop=True)

In [45]:
if "ProgramID" in para_data_df.columns:
    para_data_df = para_data_df.rename(columns={"ProgramID":"ProgID"})

if "ProgramID" in complete_data_df.columns:
    complete_data_df = complete_data_df.rename(columns={"ProgramID":"ProgID"})

## Build Batch Variables

In [47]:

if args.type_information=="para_occ":
    
    # Fields to carry from complete_data_df for every sibling variable in a batch.
    # Multi-occurrence fields (one row per paragraph occurrence) are aggregated into lists;
    # identity fields (same value for every occurrence of a given VarName in a program) are
    # kept as-is (first value after deduplication per VarName × Father × app_name group).

    BATCH_FIELDS = ['VarID', 'ParaID', 'ParaName', 'ProgID', 'ProgramName',
                    'ParaStartRow', 'ParaEndRow', 'ParaStartCol', 'ParaEndCol',
                    'PATHSTR', 'corrected_path', 'Type', 'IsField',  'PIC', 'szValues', 'iLevel', 'Father']

    # Multi-valued fields: aggregated as a list across all occurrences of a VarName.
    LIST_FIELDS  = ['ParaID', 'ParaName', 'ProgID', 'ProgramName',
                    'ParaStartRow', 'ParaEndRow', 'ParaStartCol', 'ParaEndCol', 'PATHSTR', 'corrected_path',  'PIC', 'szValues', 'iLevel', 'Father']
    
    # Single-valued fields: take the first (representative) value per VarName.
    FIRST_FIELDS = ['VarID', 'Type', 'IsField']


if args.type_information=="var_occ":
    
    # Fields to carry from complete_data_df for every sibling variable in a batch.
    # Multi-occurrence fields (one row per var occurrence) are aggregated into lists;
    # identity fields (same value for every occurrence of a given VarName in a program) are
    # kept as-is (first value after deduplication per VarName × Father × app_name group).

    BATCH_FIELDS = ['VarID', 'ProgID', 'ProgramName',
                    'VarStartRow', 'VarEndRow',
                    'PATHSTR', 'corrected_path', 'Type', 'IsField',  'PIC', 'szValues', 'iLevel', 'Father']

    # Multi-valued fields: aggregated as a list across all occurrences of a VarName.
    LIST_FIELDS  = ['ProgID', 'ProgramName',
                    'VarStartRow', 'VarEndRow', 'PATHSTR', 'corrected_path',  'Type', 'PIC', 'szValues', 'iLevel', 'Father']
    
    # Single-valued fields: take the first (representative) value per VarName.
    FIRST_FIELDS = ['VarID', 'Type', 'IsField']



def _build_sibling_entry(group_df):
    """
    Given all complete_data_df rows for one (app_name, Father, VarName) group,
    produce a dict with:
      - 'VarName'           : str
      - LIST_FIELDS entries : list (all values across occurrences)
      - FIRST_FIELDS entries: scalar (first occurrence value)
    """
    entry = {'VarName': group_df['VarName'].iloc[0]}
    for col in LIST_FIELDS:
        entry[col] = group_df[col].tolist()
    for col in FIRST_FIELDS:
        entry[col] = group_df[col].iloc[0]
    return entry

# Pre-build sibling_map: (app_name, Father) -> list of sibling dicts
# Each dict contains VarName + all BATCH_FIELDS (list or scalar as above).
sibling_map = {}
for (app_name_key, father_key), grp in complete_data_df.groupby(['app_name', 'Father']):
    siblings = []
    for var_name_key, var_grp in grp.groupby('VarName'):
        siblings.append(_build_sibling_entry(var_grp))
    sibling_map[(app_name_key, father_key)] = siblings

In [48]:
def get_batch_for_variable(var_name, app_name, father_list):
    """
    Build the batch variable list for a single variable from para_data_df.

    Returns a list of dicts — one per unique sibling variable (including the
    target variable itself, always first).  Each dict contains:
        VarName        : str
        VarID          : scalar (first occurrence)
        Type           : scalar (first occurrence)
        IsField        : scalar (first occurrence)
        ParaID         : list   (all occurrences)
        ParaName       : list   (all occurrences)
        ProgID         : list   (all occurrences)
        ProgramName    : list   (all occurrences)
        ParaStartRow   : list   (all occurrences)
        ParaEndRow     : list   (all occurrences)
        ParaStartCol   : list   (all occurrences)
        ParaEndCol     : list   (all occurrences)
        PATHSTR        : list   (all occurrences)
        corrected_path : list   (all occurrences)
        'PIC'          : list   (all occurrences) 
        'szValues'     : list   (all occurrences)
        'iLevel'       : list   (all occurrences) 
        'Father'       : list   (all occurrences)

    Args:
        var_name   : str  – the variable under consideration
        app_name   : str  – application name (used to scope the lookup)
        father_list: list – list of Father values from para_data_df for this variable

    Returns:
        list[dict]  (var_name entry first, then siblings)
    """
    # Normalise: accept both a scalar and a list
    if not isinstance(father_list, (list, tuple)):
        father_list = [father_list]

    seen_names = set()   # deduplication guard
    batch = []

    # Collect all sibling dicts across every unique Father
    unique_fathers = set(father_list)
    for father in unique_fathers:
        try:
            if father is None or (not isinstance(father, str) and father < 0):
                continue
        except TypeError:
            continue

        for sibling_dict in sibling_map.get((app_name, father), []):
            sname = sibling_dict['VarName']
            if sname not in seen_names:
                seen_names.add(sname)
                batch.append(sibling_dict)

    # Ensure the target variable is first; if it was not found in complete_data_df
    # (e.g. root-level / no matching Father), insert a minimal placeholder entry.
    target_entry = next((d for d in batch if d['VarName'] == var_name), None)
    if target_entry is None:
        # Build a minimal entry from complete_data_df directly
        var_rows = complete_data_df[
            (complete_data_df['VarName'] == var_name) &
            (complete_data_df['app_name'] == app_name)
        ]
        if not var_rows.empty:
            target_entry = _build_sibling_entry(var_rows)
        else:
            target_entry = {'VarName': var_name}
        batch.insert(0, target_entry)
    elif batch[0]['VarName'] != var_name:
        batch.remove(target_entry)
        batch.insert(0, target_entry)

    return batch

In [49]:
batch_variables_map = {}  # key: original para_data_df index -> batch list

# Iterate over every unique application present in para_data_df
applications = sorted(para_data_df['app_name'].unique())

for app_name in applications:
    current_para_df = para_data_df[para_data_df['app_name'] == app_name]
    app_complete_df = complete_data_df[complete_data_df['app_name'] == app_name]

    # Quick sanity-check stats (mirrors the original notebook)
    print(
        app_name,
        len(current_para_df),
        "IsField:",
        Counter(
            v
            for vals in current_para_df['IsField']
            for v in (vals if isinstance(vals, list) else [vals])
        )
    )

    for original_idx, row in current_para_df.iterrows():
        var_name = row[args.variable_column]
        father_list = row[args.batch_criteria]  # list of Father values for all occurrences

        batch_variables_map[original_idx] = get_batch_for_variable(
            var_name=var_name,
            app_name=app_name,
            father_list=father_list
        )

# Attach batch_variables back onto para_data_df using original index
para_data_df['batch_variables'] = para_data_df.index.map(batch_variables_map)

CARDDEMO 59 IsField: Counter({-1: 857, 0: 29})
CRYPTOCOB 20 IsField: Counter({-1: 213})
DEBINIX 17 IsField: Counter({-1: 231})
ETALAB 67 IsField: Counter({-1: 273, 0: 4})
GENAPP 87 IsField: Counter({-1: 321, 0: 7})
Z390DEV 36 IsField: Counter({-1: 6160, 0: 20})
ZECS 14 IsField: Counter({-1: 90, 0: 31})


## Merge with Annotation Set and Save

In [51]:
annotation_df = pd.read_json(args.annotation_set)

In [52]:
# Keep only the columns needed for the merge and downstream use.
# batch_variables is now a list of dicts: each dict has VarName + BATCH_FIELDS.
batch_df = para_data_df[['variable_name', 'app_name', 'batch_variables']].copy()

# para_data_df uses 'variable_name'; annotation_df is expected to have at least
# 'variable_name' and 'app_name'.
record_level_df = pd.merge(
    annotation_df,
    batch_df,
    on=[args.variable_column, 'app_name'],
    how='inner'
)

In [54]:
saving_path = f"../data/batched_data/record/batched_data_{args.batch_method}_25June2026_{args.type_information}.csv"
saving_path

'../data/batched_data/record/batched_data_record_25June2026_var_occ.csv'

In [55]:
os.makedirs(os.path.dirname(saving_path), exist_ok=True)
record_level_df.to_csv(saving_path, index=False)